# TRIAGE-RTL on RTLLM-2.0 - training-free Kaggle inference

This notebook evaluates a **robust inference-time pipeline on top of the released ChipMATE Verilog/Python agents**. It does **not** update any model weights. The two gated checkpoints are downloaded directly from Hugging Face at exact revisions. The deliverable notebook contains no credential; use a private Kaggle `HF_TOKEN` secret or a temporary private execution copy.

### What is different from vanilla ChipMATE inference
- full **Verilog x Python** candidate matrix instead of only index-paired candidates;
- shared multi-seed + directed stimulus bank and a growing regression bank;
- conservative **spec-only consensus witnesses** from both agents;
- responsibility routing: repair **Verilog**, **Python**, or **both**;
- counterexample minimization + deterministic Yosys/static localization before Verilog repair;
- evidence-gated non-regression + backtracking;
- hard limits on repair rounds and LLM completions;
- **official RTLLM testbench is blind**: it is used only after the candidate is frozen;
- atomic checkpointing after repair rounds and completed problems.

This remains research code. Agreement and consensus witnesses are evidence, not formal proof; compare against unmodified ChipMATE under the same model, completion budget, and evaluator before making a performance claim.

**Kaggle hardware:** T4 x2, with one ChipMATE-4B agent per GPU.


In [ ]:
# 1) Reproducible environment setup - Internet must be ON in Kaggle.
import os, sys, subprocess, pathlib, shutil, time
from pathlib import Path

# The experiment's time budget includes package install, repository setup, and
# local model-server startup, not only benchmark inference.
TRIAGE_NOTEBOOK_START_EPOCH = time.time()

WORK = Path('/kaggle/working')
CHIPMATE = WORK / 'ChipMATE'
RTLLM = WORK / 'RTLLM'
CHIPMATE_COMMIT = 'd94087b5159aea482648b26398a2ffb164246671'
RTLLM_COMMIT = '41b26896e33b536940116a975626455eed3de65e'

# Keep the large model cache on Kaggle's root filesystem. `/kaggle/working`
# has a much smaller saved-output quota and must remain available for results.
os.environ['HF_HOME'] = '/root/.cache/huggingface'
os.environ['HF_HUB_CACHE'] = '/root/.cache/huggingface/hub'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['PIP_NO_CACHE_DIR'] = '1'

# A temporary private execution copy may set HF_TOKEN before this cell. The
# reusable notebook instead reads the same value from Kaggle Secrets.
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN').strip()
    except Exception as exc:
        raise RuntimeError(
            'The gated ChipMATE checkpoints require a private Kaggle secret '
            'named HF_TOKEN.'
        ) from exc
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is empty.')
os.environ['HF_TOKEN'] = HF_TOKEN


def disk_report(label):
    root = shutil.disk_usage('/')
    work = shutil.disk_usage('/kaggle/working')
    print(
        f'{label}: root_free={root.free/2**30:.1f} GiB, '
        f'working_free={work.free/2**30:.1f} GiB'
    )


def checkout_exact(repo, url, commit):
    if not (repo / '.git').exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', url, str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', commit], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', '--force', commit], check=True)


disk_report('Before setup')

# System tools used by verification/localization.
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'iverilog', 'yosys', 'git'], check=True)

# Qwen3.5 requires a recent vLLM. Pin the reviewed release and avoid pip's
# wheel cache so a future package update cannot silently change the runtime.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', '-U',
    'vllm==0.25.1', 'openai>=1.40', 'huggingface_hub>=0.25', 'requests>=2.31'
], check=True)
import vllm
print('vLLM:', vllm.__version__)

checkout_exact(CHIPMATE, 'https://github.com/zhongkaiyu/ChipMATE.git', CHIPMATE_COMMIT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', '-e', str(CHIPMATE)], check=True)
# Editable-install .pth files are processed at interpreter startup. Kaggle keeps
# this notebook interpreter alive, so expose the pinned source tree explicitly.
if str(CHIPMATE) not in sys.path:
    sys.path.insert(0, str(CHIPMATE))
import chipmate
print('ChipMATE import:', Path(chipmate.__file__).resolve())
checkout_exact(RTLLM, 'https://github.com/hkust-zhiyao/RTLLM.git', RTLLM_COMMIT)

print('iverilog:', subprocess.check_output(['iverilog', '-V'], stderr=subprocess.STDOUT, text=True).splitlines()[0])
print('yosys:', subprocess.check_output(['yosys', '-V'], text=True).strip())
print('ChipMATE commit:', subprocess.check_output(['git', '-C', str(CHIPMATE), 'rev-parse', 'HEAD'], text=True).strip())
print('RTLLM commit:', subprocess.check_output(['git', '-C', str(RTLLM), 'rev-parse', 'HEAD'], text=True).strip())
disk_report('After setup')


In [ ]:
# 2) Download exact ChipMATE revisions and verify both GPUs.
import concurrent.futures
import json, os, shutil, torch
from pathlib import Path
from huggingface_hub import snapshot_download

V_REPO = 'core12345/ChipMATE-V-4B'
P_REPO = 'core12345/ChipMATE-P-4B'
V_REVISION = '7fbb17249a49d109787bbc024a7a21b83302a1c2'
P_REVISION = 'cf59695163ad3e4b63bcc6c4eaa91b05fedc4a2e'
EXPECTED_WEIGHT_BYTES = 9_078_620_536


def download_pinned(repo, revision):
    return Path(snapshot_download(
        repo_id=repo,
        revision=revision,
        token=HF_TOKEN,
        cache_dir=os.environ['HF_HUB_CACHE'],
        max_workers=8,
    ))


print('Downloading the two revision-pinned gated checkpoints in parallel...')
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
    v_future = pool.submit(download_pinned, V_REPO, V_REVISION)
    p_future = pool.submit(download_pinned, P_REPO, P_REVISION)
    V_MODEL_PATH = v_future.result()
    P_MODEL_PATH = p_future.result()


def validate_model(role, model_dir, repo, revision):
    weights = list(model_dir.glob('*.safetensors'))
    if not weights:
        raise FileNotFoundError(f'{role} checkpoint has no safetensors weights: {model_dir}')
    total = sum(p.stat().st_size for p in weights)
    if total != EXPECTED_WEIGHT_BYTES:
        raise RuntimeError(
            f'{role} weight-size mismatch: expected {EXPECTED_WEIGHT_BYTES}, got {total}'
        )
    config = json.loads((model_dir / 'config.json').read_text())
    print(
        f'{role} model: {model_dir} | repo={repo} | revision={revision} | '
        f'{total/2**30:.2f} GiB | '
        f'architectures={config.get("architectures")}'
    )


validate_model('V', V_MODEL_PATH, V_REPO, V_REVISION)
validate_model('P', P_MODEL_PATH, P_REPO, P_REVISION)
if V_MODEL_PATH == P_MODEL_PATH:
    raise RuntimeError('Verilog and Python agents resolved to the same checkpoint path.')

# The vLLM servers load local snapshots, so the credential is no longer needed.
os.environ.pop('HF_TOKEN', None)
HF_TOKEN = None
print(f'Root filesystem free after downloads: {shutil.disk_usage("/").free/2**30:.1f} GiB')

ngpu = torch.cuda.device_count()
print('GPU count:', ngpu)
for i in range(ngpu):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, f'{p.total_memory/2**30:.1f} GiB')

if ngpu < 2:
    raise RuntimeError(
        'This run needs two GPUs so the Verilog and Python agents stay separate. '
        'Select Kaggle T4 x2 (or a two-GPU machine) and run again.'
    )


In [ ]:
# 3) Start the downloaded ChipMATE agents as two local vLLM servers.
import os, sys, time, signal, subprocess, requests, shutil
from pathlib import Path

V_MODEL = 'triage-chipmate-verilog'
P_MODEL = 'triage-chipmate-python'

SERVER_DIR = Path('/kaggle/working/vllm_logs')
SERVER_DIR.mkdir(exist_ok=True)

# Kill servers started by an earlier execution of this cell.
pidfile = SERVER_DIR / 'pids.txt'
if pidfile.exists():
    for x in pidfile.read_text().split():
        try:
            os.kill(int(x), signal.SIGTERM)
        except Exception:
            pass
    time.sleep(3)


def launch(model_path, served_name, gpu, port, log_name, max_len=8192):
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    env['TOKENIZERS_PARALLELISM'] = 'false'
    log = open(SERVER_DIR / log_name, 'w')
    cmd = [
        sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
        '--model', str(model_path), '--served-model-name', served_name,
        '--host', '127.0.0.1', '--port', str(port),
        '--dtype', 'half',
        '--max-model-len', str(max_len),
        '--gpu-memory-utilization', '0.90',
        '--max-num-seqs', '4',
    ]
    return subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log


def ready(port):
    try:
        r = requests.get(f'http://127.0.0.1:{port}/v1/models', timeout=3)
        return r.status_code == 200
    except Exception:
        return False


pv, lv = launch(V_MODEL_PATH, V_MODEL, 0, 8001, 'verilog.log')
pp, lp = launch(P_MODEL_PATH, P_MODEL, 1, 8002, 'python.log')
pidfile.write_text(f'{pv.pid}\n{pp.pid}\n')

# The checkpoints are now local snapshots; this phase only loads them onto GPUs.
deadline = time.time() + 20*60
while time.time() < deadline:
    if ready(8001) and ready(8002):
        break
    if pv.poll() is not None or pp.poll() is not None:
        break
    time.sleep(10)

if not (ready(8001) and ready(8002)):
    print('Verilog server tail:')
    print((SERVER_DIR / 'verilog.log').read_text(errors='replace')[-8000:])
    print('Python server tail:')
    print((SERVER_DIR / 'python.log').read_text(errors='replace')[-8000:])
    raise RuntimeError('One or both vLLM servers failed to start. Inspect the log tails above.')

print('V server ready:', V_MODEL, 'port 8001', 'path', V_MODEL_PATH)
print('P server ready:', P_MODEL, 'port 8002', 'path', P_MODEL_PATH)
print(f'Root filesystem free after model startup: {shutil.disk_usage("/").free/2**30:.1f} GiB')


In [ ]:
%%writefile /kaggle/working/triage_rtl_kaggle.py
"""
TRIAGE-RTL: training-free, responsibility-aware RTL inference on top of ChipMATE.

Designed for Kaggle with 2 GPUs (2xT4 is sufficient for the 4B pair).
No model weights are updated. Official RTLLM testbenches are used ONLY after
inference, never as repair feedback.

Main safeguards over vanilla ChipMATE inference:
  1) full V x P cross-product scoring rather than index-paired scoring,
  2) multiple deterministic stimulus families + a growing regression bank,
  3) spec-only consensus witness tests generated independently by both agents,
  4) responsibility routing (repair Verilog, Python, or both),
  5) deterministic Yosys/static localization before LLM repair,
  6) counterexample minimization,
  7) evidence-gated backtracking / non-regression,
  8) hard round and LLM-completion budgets,
  9) atomic checkpointing after every problem and every repair round,
 10) blind official RTLLM evaluation only after the final candidate is frozen.
"""
from __future__ import annotations

import ast
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

# ------------------------------ Configuration ------------------------------

@dataclass
class Config:
    # Repositories / output
    rtllm_root: str = "/kaggle/working/RTLLM"
    out_dir: str = "/kaggle/working/triage_rtllm"
    resume_from: Optional[str] = None

    # Local OpenAI-compatible endpoints serving ChipMATE models
    v_model: str = "core12345/ChipMATE-V-4B"
    p_model: str = "core12345/ChipMATE-P-4B"
    v_base_url: str = "http://127.0.0.1:8001/v1"
    p_base_url: str = "http://127.0.0.1:8002/v1"

    # Inference budget. Vanilla ChipMATE paper protocol with n=3,t=3 samples
    # 9 Verilog + 9 Python completions = 18 completions/problem.
    fair_completion_budget: int = 18
    k_initial: int = 3
    max_repair_rounds: int = 3
    code_temperature: float = 1.0
    repair_temperature: float = 0.8
    code_max_tokens: int = 3072
    witness_max_tokens: int = 1800

    # Verification. We simulate each V and each P only once on a shared bank,
    # then compute the full cross-product matrix from cached outputs.
    random_seeds: Tuple[int, ...] = (41, 137, 271)
    random_per_seed: int = 96
    directed_limit: int = 48
    fresh_random_per_round: int = 64
    max_regression_vectors: int = 512

    # Acceptance thresholds
    min_improvement: float = 0.001
    route_margin: float = 0.06
    strong_pair_threshold: float = 0.995
    strong_support_threshold: float = 0.965
    witness_weight: float = 0.18

    # Witness tests generated from specification only
    canonical_witness_sequences: int = 10
    max_consensus_witnesses: int = 6

    # Counterexample / localization budgets
    ddmin_calls: int = 8
    yosys_depth: int = 7
    max_localized_lines: int = 90

    # Kaggle wall-clock guard. Kaggle limits can vary; stopping early gives time
    # for checkpoint flush and notebook output commit.
    session_budget_hours: float = 11.25
    stop_margin_minutes: float = 25.0
    session_start_epoch: Optional[float] = None

    # Abort systemic failures early instead of recording the same infrastructure
    # error for every benchmark task.
    max_consecutive_pipeline_errors: int = 3

    # Evaluation
    official_eval_timeout_sec: int = 90
    workers: int = 4


CFG = Config()


# ------------------------------ Utilities ------------------------------

def sha(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="replace")).hexdigest()[:16]


def norm_ws(s: str) -> str:
    return " ".join((s or "").split())


def atomic_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)


def atomic_jsonl_append(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    # JSONL append itself is atomic enough for a single process; fsync reduces
    # the chance of losing the just-completed problem on abrupt shutdown.
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def strip_code_fences(text: str) -> str:
    t = (text or "").strip()
    t = re.sub(r"^```(?:json|python|verilog|systemverilog)?\s*", "", t, flags=re.I)
    t = re.sub(r"\s*```$", "", t)
    return t.strip()


def first_json_object(text: str) -> Optional[dict]:
    """Extract first balanced JSON object; tolerant of prose/code fences."""
    s = strip_code_fences(text)
    start = s.find("{")
    if start < 0:
        return None
    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    chunk = s[start:i + 1]
                    try:
                        return json.loads(chunk)
                    except Exception:
                        return None
    return None


def dedup_codes(codes: Iterable[str]) -> List[str]:
    seen, out = set(), []
    for c in codes:
        c = (c or "").strip()
        if not c:
            continue
        key = sha(re.sub(r"\s+", " ", c))
        if key not in seen:
            seen.add(key)
            out.append(c)
    return out


def module_name_from_sv(sv: str) -> Optional[str]:
    m = re.search(r"\bmodule\s+([A-Za-z_]\w*)", sv or "")
    return m.group(1) if m else None


def rename_first_module(sv: str, target: str) -> str:
    m = re.search(r"\bmodule\s+([A-Za-z_]\w*)", sv or "")
    if not m:
        return sv
    old = m.group(1)
    out = re.sub(r"\bmodule\s+" + re.escape(old) + r"\b", f"module {target}", sv, count=1)
    out = re.sub(r"endmodule\s*:\s*" + re.escape(old) + r"\b", f"endmodule : {target}", out)
    return out


def extract_header_stub(sv: str) -> str:
    """Keep only first module declaration/port list; never implementation logic."""
    m = re.search(r"\bmodule\s+[A-Za-z_]\w*", sv)
    if not m:
        raise ValueError("No module declaration found in reference RTL")
    i = m.start()
    # Find semicolon terminating module declaration while respecting parentheses.
    depth = 0
    saw_paren = False
    in_str = False
    esc = False
    j = m.end()
    for j in range(m.end(), len(sv)):
        ch = sv[j]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == "(":
            depth += 1
            saw_paren = True
        elif ch == ")":
            depth = max(0, depth - 1)
        elif ch == ";" and (not saw_paren or depth == 0):
            return sv[i:j + 1] + "\nendmodule\n"
    raise ValueError("Could not isolate module header")


# ------------------------------ Session guard ------------------------------

class TimeBudgetExceeded(RuntimeError):
    pass


class SessionGuard:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.t0 = time.monotonic()
        self.elapsed_before_start = max(
            0.0,
            time.time() - cfg.session_start_epoch if cfg.session_start_epoch else 0.0,
        )
        self.problem_times: List[float] = []

    def elapsed(self) -> float:
        return self.elapsed_before_start + time.monotonic() - self.t0

    def remaining(self) -> float:
        return self.cfg.session_budget_hours * 3600 - self.elapsed()

    def record_problem(self, seconds: float) -> None:
        self.problem_times.append(seconds)

    def predicted_next(self) -> float:
        if not self.problem_times:
            return 20 * 60  # conservative first estimate
        recent = self.problem_times[-5:]
        return max(5 * 60, sum(recent) / len(recent) * 1.25)

    def check(self, label: str = "work", include_next_problem: bool = False) -> None:
        margin = self.cfg.stop_margin_minutes * 60
        need = margin + (self.predicted_next() if include_next_problem else 0)
        if self.remaining() <= need:
            raise TimeBudgetExceeded(
                f"Stopping before {label}: {self.remaining()/60:.1f} min remain; "
                f"reserve={need/60:.1f} min"
            )


# ------------------------------ RTLLM discovery ------------------------------

@dataclass
class Task:
    task_id: str
    folder: str
    question: str
    ref_stub: str
    target_module: str
    testbench_path: str
    reference_path: str


def find_reference_sv(folder: Path) -> Path:
    # Preferred names first.
    preferred = [
        "verified_verilog.v", "verified_RTL.v", "designer_RTL.v", "reference.v",
        "ref.v", "golden.v"
    ]
    for name in preferred:
        p = folder / name
        if p.exists():
            return p
    candidates = []
    for p in folder.glob("*.v"):
        low = p.name.lower()
        if "testbench" in low or low.startswith("tb") or "generated" in low:
            continue
        candidates.append(p)
    if not candidates:
        raise FileNotFoundError(f"No reference RTL found in {folder}")
    # Prefer a filename containing verified/designer/ref.
    candidates.sort(key=lambda p: (
        0 if any(k in p.name.lower() for k in ("verified", "designer", "reference", "ref")) else 1,
        len(p.name), p.name
    ))
    return candidates[0]


def discover_rtllm(root: str) -> List[Task]:
    rootp = Path(root)
    tasks: List[Task] = []
    for desc in rootp.rglob("design_description.txt"):
        folder = desc.parent
        tb = folder / "testbench.v"
        if not tb.exists():
            continue
        try:
            ref_path = find_reference_sv(folder)
            ref_text = ref_path.read_text(encoding="utf-8", errors="replace")
            stub = extract_header_stub(ref_text)
            mod = module_name_from_sv(stub)
            if not mod:
                continue
            question = desc.read_text(encoding="utf-8", errors="replace")
            rel = str(folder.relative_to(rootp))
            # Use the relative path, not only the leaf folder name, so checkpoint IDs
            # remain unique even if benchmark categories reuse a design name.
            task_id = re.sub(r"[^A-Za-z0-9_.-]+", "__", rel).strip("_") or folder.name
            tasks.append(Task(
                task_id=task_id,
                folder=rel,
                question=question,
                ref_stub=stub,
                target_module=mod,
                testbench_path=str(tb),
                reference_path=str(ref_path),
            ))
        except Exception as e:
            print(f"[WARN] skip {folder}: {e}")
    tasks.sort(key=lambda t: (t.folder, t.task_id))
    return tasks


# ------------------------------ ChipMATE imports ------------------------------

def import_chipmate():
    from chipmate import make_backend
    from chipmate.prompts import build_v_prompt, build_p_prompt, extract_verilog, extract_python
    from chipmate.cross_verify import (
        parse_ports, gen_stimuli, simulate_verilog, simulate_python, is_reset, is_active_low
    )
    return {
        "make_backend": make_backend,
        "build_v_prompt": build_v_prompt,
        "build_p_prompt": build_p_prompt,
        "extract_verilog": extract_verilog,
        "extract_python": extract_python,
        "parse_ports": parse_ports,
        "gen_stimuli": gen_stimuli,
        "simulate_verilog": simulate_verilog,
        "simulate_python": simulate_python,
        "is_reset": is_reset,
        "is_active_low": is_active_low,
    }


# ------------------------------ Stimulus bank ------------------------------

def mask(width: int, x: int) -> int:
    return int(x) & ((1 << max(1, width)) - 1)


def canonical_values(width: int) -> List[int]:
    m = (1 << width) - 1
    vals = [0, 1, m]
    if m > 0:
        vals += [max(0, m - 1)]
    if width > 1:
        vals += [1 << (width - 1), (1 << (width - 1)) - 1]
    # Alternating bit patterns.
    a = sum(1 << b for b in range(width) if b % 2 == 1)
    b = sum(1 << b for b in range(width) if b % 2 == 0)
    vals += [a, b]
    out = []
    for v in vals:
        v = mask(width, v)
        if v not in out:
            out.append(v)
    return out


def build_directed_vectors(inputs: Sequence[Tuple[str, int]], is_reset_fn, limit: int) -> List[Dict[str, int]]:
    data = [(n, w) for n, w in inputs if n.lower() != "clk" and not is_reset_fn(n)]
    if not data:
        return [{}]
    zeros = {n: 0 for n, _ in data}
    maxs = {n: (1 << w) - 1 for n, w in data}
    vecs: List[Dict[str, int]] = [zeros, maxs]

    # Vary one signal at a time around canonical boundary values.
    for n, w in data:
        for v in canonical_values(w):
            x = dict(zeros)
            x[n] = v
            vecs.append(x)
            if len(vecs) >= limit:
                break
        if len(vecs) >= limit:
            break

    # Cross-boundary patterns without full Cartesian explosion.
    idx = 0
    while len(vecs) < limit:
        x = {}
        for j, (n, w) in enumerate(data):
            vals = canonical_values(w)
            x[n] = vals[(idx + j) % len(vals)]
        vecs.append(x)
        idx += 1
        if idx > limit * 2:
            break

    # Dedup.
    seen, out = set(), []
    for v in vecs:
        key = tuple(sorted(v.items()))
        if key not in seen:
            seen.add(key)
            out.append(v)
    return out[:limit]


def build_regression_bank(cm: dict, ref_stub: str, cfg: Config, round_count: int = 0) -> List[Dict[str, int]]:
    inputs, _ = cm["parse_ports"](ref_stub)
    stims: List[Dict[str, int]] = []
    stims += build_directed_vectors(inputs, cm["is_reset"], cfg.directed_limit)
    for s in cfg.random_seeds:
        stims += cm["gen_stimuli"](inputs, cfg.random_per_seed, seed=s)
    for r in range(round_count):
        stims += cm["gen_stimuli"](
            inputs, cfg.fresh_random_per_round, seed=10007 + 7919 * r
        )
    # Dedup while keeping order.
    seen, out = set(), []
    for v in stims:
        key = tuple(sorted(v.items()))
        if key not in seen:
            seen.add(key)
            out.append(v)
            if len(out) >= cfg.max_regression_vectors:
                break
    return out


# ------------------------------ Evidence matrix ------------------------------

@dataclass
class SimResult:
    outputs: Optional[List[Dict[str, int]]]
    error: Optional[str]


@dataclass
class MatrixEvidence:
    matrix: List[List[float]]
    v_support: List[float]
    p_support: List[float]
    v_sims: List[SimResult]
    p_sims: List[SimResult]
    best_i: int
    best_j: int
    best_pair_match: float
    best_score_no_witness: float
    first_mismatch: Optional[Dict[str, Any]]


def compare_rows(vrows: List[Dict[str, int]], prows: List[Dict[str, int]], outputs: Sequence[Tuple[str, int]]) -> float:
    total = 0
    bad = 0
    for vr, pr in zip(vrows, prows):
        for name, _ in outputs:
            total += 1
            if vr.get(name) != pr.get(name):
                bad += 1
    return 1.0 - bad / total if total else 0.0


def first_mismatch_from_rows(
    vrows: List[Dict[str, int]], prows: List[Dict[str, int]],
    outputs: Sequence[Tuple[str, int]], stimuli: Sequence[Dict[str, int]]
) -> Optional[Dict[str, Any]]:
    for idx, (vr, pr) in enumerate(zip(vrows, prows)):
        for name, _ in outputs:
            if vr.get(name) != pr.get(name):
                return {
                    "test": idx,
                    "signal": name,
                    "verilog": vr.get(name),
                    "python": pr.get(name),
                    "inputs": stimuli[idx] if idx < len(stimuli) else {},
                }
    return None


def topk_mean(xs: Sequence[float], k: int = 2) -> float:
    vals = sorted([x for x in xs if x >= 0.0], reverse=True)
    if not vals:
        return 0.0
    vals = vals[: min(k, len(vals))]
    return sum(vals) / len(vals)


def evaluate_matrix(
    cm: dict, ref_stub: str, v_pool: Sequence[str], p_pool: Sequence[str],
    stimuli: Sequence[Dict[str, int]], workers: int = 4
) -> MatrixEvidence:
    inputs, outputs = cm["parse_ports"](ref_stub)
    has_clk = any(n.lower() == "clk" for n, _ in inputs)
    resets = [(n, w, cm["is_active_low"](n)) for n, w in inputs
              if cm["is_reset"](n) and n.lower() != "clk"]

    def sim_v(code: str) -> SimResult:
        try:
            rows, err = cm["simulate_verilog"](code, inputs, outputs, list(stimuli), has_clk, resets)
            return SimResult(rows, err)
        except Exception as e:
            return SimResult(None, f"verilog_exception: {e}")

    def sim_p(code: str) -> SimResult:
        try:
            rows, err = cm["simulate_python"](code, inputs, outputs, list(stimuli), has_clk, resets)
            return SimResult(rows, err)
        except Exception as e:
            return SimResult(None, f"python_exception: {e}")

    v_sims: List[Optional[SimResult]] = [None] * len(v_pool)
    p_sims: List[Optional[SimResult]] = [None] * len(p_pool)
    jobs = {}
    with ThreadPoolExecutor(max_workers=max(1, workers)) as ex:
        for i, c in enumerate(v_pool):
            jobs[ex.submit(sim_v, c)] = ("v", i)
        for j, c in enumerate(p_pool):
            jobs[ex.submit(sim_p, c)] = ("p", j)
        for fut in as_completed(jobs):
            kind, idx = jobs[fut]
            try:
                r = fut.result()
            except Exception as e:
                r = SimResult(None, f"worker_exception: {e}")
            if kind == "v":
                v_sims[idx] = r
            else:
                p_sims[idx] = r

    v_sims2 = [x or SimResult(None, "missing") for x in v_sims]
    p_sims2 = [x or SimResult(None, "missing") for x in p_sims]

    matrix = [[0.0 for _ in p_pool] for _ in v_pool]
    for i, vs in enumerate(v_sims2):
        for j, ps in enumerate(p_sims2):
            if vs.outputs is not None and ps.outputs is not None and not vs.error and not ps.error:
                matrix[i][j] = compare_rows(vs.outputs, ps.outputs, outputs)

    v_support = [topk_mean(row, 2) for row in matrix]
    p_support = [topk_mean([matrix[i][j] for i in range(len(v_pool))], 2)
                 for j in range(len(p_pool))]

    best_i, best_j, best_score, best_pair = 0, 0, -1.0, -1.0
    for i in range(len(v_pool)):
        for j in range(len(p_pool)):
            pair = matrix[i][j]
            # Cross-product consensus score: avoid selecting a coincidental pair
            # that agrees with each other but not with the rest of the pool.
            score = 0.62 * pair + 0.19 * v_support[i] + 0.19 * p_support[j]
            if score > best_score:
                best_i, best_j, best_score, best_pair = i, j, score, pair

    fm = None
    if v_pool and p_pool:
        vs, ps = v_sims2[best_i], p_sims2[best_j]
        if vs.outputs is not None and ps.outputs is not None:
            fm = first_mismatch_from_rows(vs.outputs, ps.outputs, outputs, stimuli)

    return MatrixEvidence(
        matrix=matrix,
        v_support=v_support,
        p_support=p_support,
        v_sims=v_sims2,
        p_sims=p_sims2,
        best_i=best_i,
        best_j=best_j,
        best_pair_match=best_pair,
        best_score_no_witness=best_score,
        first_mismatch=fm,
    )


# ------------------------------ Spec-only consensus witnesses ------------------------------

@dataclass
class Witness:
    test_id: str
    steps: List[Dict[str, int]]
    expected: Dict[str, int]
    evidence_v: str
    evidence_p: str


def canonical_sequences(cm: dict, ref_stub: str, limit: int) -> List[Dict[str, Any]]:
    inputs, _ = cm["parse_ports"](ref_stub)
    data = [(n, w) for n, w in inputs if n.lower() != "clk" and not cm["is_reset"](n)]
    vecs = build_directed_vectors(inputs, cm["is_reset"], max(limit, 8))
    seqs = []
    # Single-step canonical cases.
    for i, v in enumerate(vecs[: max(1, limit // 2)]):
        seqs.append({"test_id": f"S{i:02d}", "steps": [v]})
    # Short sequences useful for sequential logic.
    if data:
        z = {n: 0 for n, _ in data}
        m = {n: (1 << w) - 1 for n, w in data}
        one = {n: 1 & ((1 << w) - 1) for n, w in data}
        patterns = [
            [z, one],
            [z, m],
            [one, z, one],
            [z, one, one, z],
            [m, z, m],
        ]
        for p in patterns:
            if len(seqs) >= limit:
                break
            seqs.append({"test_id": f"Q{len(seqs):02d}", "steps": p})
    return seqs[:limit]


def witness_prompt(spec: str, ref_stub: str, seqs: List[Dict[str, Any]]) -> str:
    return f"""You are a specification-only RTL verification witness.
You MUST NOT see or infer from any candidate implementation. Use only the natural-language specification below.

TASK:
For the fixed canonical input sequences, predict the output values after the FINAL step only when the value is explicitly and confidently implied by the specification. Omit ambiguous cases. Never guess.

GROUNDING RULE:
For every prediction, include `evidence_quote`: a short exact contiguous quote copied verbatim from the specification that supports the expected behavior. If you cannot provide an exact quote, omit the prediction.

INTERFACE STUB (port declarations only; no implementation logic):
{ref_stub}

SPECIFICATION:
{spec}

CANONICAL SEQUENCES:
{json.dumps(seqs, indent=2)}

Return JSON only, exactly in this shape:
{{
  "predictions": [
    {{"test_id":"S00", "expected":{{"out":0}}, "evidence_quote":"exact quote from specification"}}
  ]
}}

Rules:
- Decimal integers only.
- Use output port names exactly.
- Do not include clk/reset in expected.
- Do not invent unspecified latency, reset priority, signedness, or protocol behavior.
- It is better to return fewer high-confidence predictions than many guesses.
"""


def validate_predictions(
    obj: Optional[dict], spec: str, seqs: List[Dict[str, Any]],
    outputs: Sequence[Tuple[str, int]]
) -> Dict[str, Dict[str, Any]]:
    if not isinstance(obj, dict):
        return {}
    seq_ids = {x["test_id"] for x in seqs}
    widths = dict(outputs)
    spec_norm = norm_ws(spec)
    out = {}
    for p in obj.get("predictions", []) or []:
        if not isinstance(p, dict):
            continue
        tid = str(p.get("test_id", ""))
        exp = p.get("expected")
        quote = str(p.get("evidence_quote", "")).strip()
        if tid not in seq_ids or not isinstance(exp, dict) or not quote:
            continue
        if norm_ws(quote) not in spec_norm:
            continue
        clean = {}
        valid = True
        for name, val in exp.items():
            if name not in widths or isinstance(val, bool) or not isinstance(val, int):
                valid = False
                break
            if val < 0 or val >= (1 << widths[name]):
                valid = False
                break
            clean[name] = int(val)
        if valid and clean:
            out[tid] = {"expected": clean, "evidence_quote": quote}
    return out


def generate_consensus_witnesses(
    cm: dict, spec: str, ref_stub: str, v_backend, p_backend, cfg: Config
) -> Tuple[List[Witness], int, Dict[str, Any]]:
    inputs, outputs = cm["parse_ports"](ref_stub)
    seqs = canonical_sequences(cm, ref_stub, cfg.canonical_witness_sequences)
    prompt = witness_prompt(spec, ref_stub, seqs)
    # Independent agents, low temperature, same canonical sequences.
    with ThreadPoolExecutor(max_workers=2) as ex:
        fv = ex.submit(v_backend.generate, prompt, 1, 0.15, cfg.witness_max_tokens)
        fp = ex.submit(p_backend.generate, prompt, 1, 0.15, cfg.witness_max_tokens)
        vtxt = (fv.result() or [""])[0]
        ptxt = (fp.result() or [""])[0]
    vp = validate_predictions(first_json_object(vtxt), spec, seqs, outputs)
    pp = validate_predictions(first_json_object(ptxt), spec, seqs, outputs)
    seq_map = {x["test_id"]: x["steps"] for x in seqs}
    witnesses = []
    for tid in sorted(set(vp) & set(pp)):
        if vp[tid]["expected"] == pp[tid]["expected"]:
            witnesses.append(Witness(
                test_id=tid,
                steps=seq_map[tid],
                expected=vp[tid]["expected"],
                evidence_v=vp[tid]["evidence_quote"],
                evidence_p=pp[tid]["evidence_quote"],
            ))
            if len(witnesses) >= cfg.max_consensus_witnesses:
                break
    meta = {
        "v_valid_predictions": len(vp),
        "p_valid_predictions": len(pp),
        "consensus": len(witnesses),
    }
    return witnesses, 2, meta


def evaluate_witnesses_on_code(
    cm: dict, ref_stub: str, code: str, witnesses: Sequence[Witness], kind: str
) -> Tuple[float, List[Dict[str, Any]], Optional[str]]:
    if not witnesses:
        return 1.0, [], None
    inputs, outputs = cm["parse_ports"](ref_stub)
    has_clk = any(n.lower() == "clk" for n, _ in inputs)
    resets = [(n, w, cm["is_active_low"](n)) for n, w in inputs
              if cm["is_reset"](n) and n.lower() != "clk"]
    passed = 0
    details = []
    first_error = None
    for w in witnesses:
        try:
            if kind == "v":
                rows, err = cm["simulate_verilog"](code, inputs, outputs, w.steps, has_clk, resets)
            else:
                rows, err = cm["simulate_python"](code, inputs, outputs, w.steps, has_clk, resets)
        except Exception as e:
            rows, err = None, f"{kind}_witness_exception: {e}"
        if err or not rows:
            if first_error is None:
                first_error = err or "no witness outputs"
            details.append({"test_id": w.test_id, "pass": False, "error": err})
            continue
        final = rows[-1]
        ok = all(final.get(k) == v for k, v in w.expected.items())
        if ok:
            passed += 1
        details.append({
            "test_id": w.test_id,
            "pass": ok,
            "expected": w.expected,
            "actual": {k: final.get(k) for k in w.expected},
        })
    return passed / len(witnesses), details, first_error


# ------------------------------ Counterexample minimization ------------------------------

def pair_has_target_mismatch(
    cm: dict, ref_stub: str, v_code: str, p_code: str,
    stimuli: Sequence[Dict[str, int]], target_signal: str
) -> Tuple[bool, Optional[Dict[str, Any]]]:
    inputs, outputs = cm["parse_ports"](ref_stub)
    has_clk = any(n.lower() == "clk" for n, _ in inputs)
    resets = [(n, w, cm["is_active_low"](n)) for n, w in inputs
              if cm["is_reset"](n) and n.lower() != "clk"]
    vr, ve = cm["simulate_verilog"](v_code, inputs, outputs, list(stimuli), has_clk, resets)
    if ve or vr is None:
        return False, None
    pr, pe = cm["simulate_python"](p_code, inputs, outputs, list(stimuli), has_clk, resets)
    if pe or pr is None:
        return False, None
    for i, (a, b) in enumerate(zip(vr, pr)):
        if a.get(target_signal) != b.get(target_signal):
            return True, {
                "test": i, "signal": target_signal,
                "verilog": a.get(target_signal), "python": b.get(target_signal),
                "inputs": stimuli[i] if i < len(stimuli) else {},
            }
    return False, None


def minimize_counterexample(
    cm: dict, ref_stub: str, v_code: str, p_code: str,
    full_stimuli: Sequence[Dict[str, int]], mismatch: Optional[Dict[str, Any]],
    max_calls: int
) -> Dict[str, Any]:
    if not mismatch:
        return {"steps": [], "mismatch": None, "calls": 0}
    idx = min(int(mismatch.get("test", 0)), len(full_stimuli) - 1)
    seq = list(full_stimuli[:idx + 1])
    target = str(mismatch.get("signal"))
    if len(seq) <= 1:
        return {"steps": seq, "mismatch": mismatch, "calls": 0}

    calls = 0
    n = 2
    while len(seq) > 1 and calls < max_calls:
        chunk = max(1, math.ceil(len(seq) / n))
        reduced = False
        for start in range(0, len(seq), chunk):
            if calls >= max_calls:
                break
            trial = seq[:start] + seq[start + chunk:]
            if not trial:
                continue
            calls += 1
            ok, mm = pair_has_target_mismatch(cm, ref_stub, v_code, p_code, trial, target)
            if ok:
                seq = trial
                mismatch = mm or mismatch
                n = max(2, n - 1)
                reduced = True
                break
        if not reduced:
            if n >= len(seq):
                break
            n = min(len(seq), n * 2)
    return {"steps": seq, "mismatch": mismatch, "calls": calls}


# ------------------------------ Yosys/static localization ------------------------------

def source_lines_from_src(src: str) -> List[int]:
    # Yosys src examples: "dut.sv:12.3-18.9" or multiple entries separated by |.
    lines = []
    for m in re.finditer(r":(\d+)(?:\.\d+)?(?:-(\d+)(?:\.\d+)?)?", src or ""):
        a = int(m.group(1))
        b = int(m.group(2) or a)
        lines.extend(range(min(a, b), max(a, b) + 1))
    return lines


def lexical_localize(code: str, signal: str, max_lines: int) -> str:
    lines = code.splitlines()
    hits = [i for i, line in enumerate(lines) if re.search(r"\b" + re.escape(signal) + r"\b", line)]
    keep = set()
    for h in hits[:12]:
        for i in range(max(0, h - 4), min(len(lines), h + 5)):
            keep.add(i)
    if not keep:
        return "\n".join(lines[:max_lines])
    out = []
    for i in sorted(keep)[:max_lines]:
        out.append(f"{i+1:4d}: {lines[i]}")
    return "\n".join(out)


def yosys_localize(code: str, failing_output: str, depth: int, max_lines: int) -> Dict[str, Any]:
    mod = module_name_from_sv(code)
    if not mod or shutil.which("yosys") is None:
        return {"method": "lexical", "snippet": lexical_localize(code, failing_output, max_lines)}
    with tempfile.TemporaryDirectory() as td:
        td = Path(td)
        svp = td / "dut.sv"
        jp = td / "net.json"
        svp.write_text(code, encoding="utf-8")
        script = f"read_verilog -sv {svp}; hierarchy -top {mod}; proc; opt; write_json {jp}"
        try:
            cp = subprocess.run(["yosys", "-q", "-p", script], capture_output=True, text=True, timeout=45)
            if cp.returncode != 0 or not jp.exists():
                raise RuntimeError(cp.stderr[-1000:])
            net = json.loads(jp.read_text(encoding="utf-8"))
            m = net["modules"].get(mod)
            if not m:
                raise KeyError(mod)
            port = m.get("ports", {}).get(failing_output)
            if not port:
                raise KeyError(failing_output)
            frontier = set(port.get("bits", []))
            visited_bits = set(frontier)
            src_lines = set()
            cells = m.get("cells", {})
            for _ in range(depth):
                next_bits = set()
                changed = False
                for _, cell in cells.items():
                    conns = cell.get("connections", {})
                    dirs = cell.get("port_directions", {})
                    out_bits = set()
                    in_bits = set()
                    for pname, bits in conns.items():
                        if dirs.get(pname) == "output":
                            out_bits |= {b for b in bits if isinstance(b, int)}
                        elif dirs.get(pname) == "input":
                            in_bits |= {b for b in bits if isinstance(b, int)}
                    if frontier & out_bits:
                        changed = True
                        next_bits |= in_bits
                        for ln in source_lines_from_src(str(cell.get("attributes", {}).get("src", ""))):
                            src_lines.add(ln)
                next_bits -= visited_bits
                if not changed or not next_bits:
                    break
                visited_bits |= next_bits
                frontier = next_bits

            lines = code.splitlines()
            keep = set()
            for ln in src_lines:
                for i in range(max(1, ln - 2), min(len(lines), ln + 2) + 1):
                    keep.add(i)
            if not keep:
                return {"method": "lexical", "snippet": lexical_localize(code, failing_output, max_lines)}
            snippet = "\n".join(f"{i:4d}: {lines[i-1]}" for i in sorted(keep)[:max_lines])
            return {"method": "yosys_backward_slice", "snippet": snippet, "source_lines": sorted(src_lines)}
        except Exception as e:
            return {
                "method": "lexical_fallback",
                "error": str(e),
                "snippet": lexical_localize(code, failing_output, max_lines),
            }


# ------------------------------ Routing / repair ------------------------------

def route_responsibility(
    ev: MatrixEvidence,
    v_witness: float,
    p_witness: float,
    witnesses_exist: bool,
    cfg: Config,
) -> str:
    i, j = ev.best_i, ev.best_j
    vs, ps = ev.v_sims[i], ev.p_sims[j]
    if vs.error and not ps.error:
        return "verilog"
    if ps.error and not vs.error:
        return "python"
    if vs.error and ps.error:
        return "both"

    # Spec-only witnesses get priority because they are generated without seeing
    # candidate code and require two-agent agreement + exact spec quotations.
    if witnesses_exist:
        if p_witness - v_witness >= 0.25:
            return "verilog"
        if v_witness - p_witness >= 0.25:
            return "python"
        if v_witness < 0.5 and p_witness < 0.5:
            return "both"

    v_sup = ev.v_support[i]
    p_sup = ev.p_support[j]
    if p_sup - v_sup >= cfg.route_margin:
        return "verilog"
    if v_sup - p_sup >= cfg.route_margin:
        return "python"
    return "both"


def combined_score(ev: MatrixEvidence, v_w: float, p_w: float, witnesses_exist: bool, cfg: Config) -> float:
    base = ev.best_score_no_witness
    if not witnesses_exist:
        return base
    w = cfg.witness_weight
    return (1.0 - w) * base + w * ((v_w + p_w) / 2.0)


def format_mismatch(mm: Optional[Dict[str, Any]], minimized: Dict[str, Any]) -> str:
    if not mm:
        return "No concrete V/P mismatch was captured; use the compile/runtime or spec-witness evidence instead."
    steps = minimized.get("steps", [])
    return (
        f"First disagreement: signal={mm.get('signal')}, Verilog={mm.get('verilog')}, "
        f"Python-peer={mm.get('python')}, input_at_failure={mm.get('inputs')}.\n"
        f"Minimized reproducing input sequence ({len(steps)} steps):\n{json.dumps(steps, indent=2)}"
    )


def witness_feedback(details: List[Dict[str, Any]]) -> str:
    bad = [d for d in details if not d.get("pass")]
    if not bad:
        return "All available spec-only consensus witness tests pass."
    return "Spec-only consensus witness failures:\n" + json.dumps(bad[:4], indent=2)


def build_verilog_repair_prompt(
    spec: str, code: str, mismatch_text: str, localization: Dict[str, Any],
    witness_text: str, compile_error: Optional[str]
) -> str:
    if compile_error:
        task = f"Fix the Verilog compile/simulation problem first: {compile_error}"
    else:
        task = "Repair the behavioral bug indicated by the evidence below."
    return f"""You are the Verilog repair agent in a code-blind verification workflow.
{task}

NON-NEGOTIABLE RULES:
- The natural-language specification is authoritative.
- You cannot see the Python peer implementation; do not blindly copy peer behavior.
- Preserve the module interface exactly.
- Make the smallest causally justified change; do not rewrite unrelated working logic.
- Preserve already-correct behavior and synthesizability.
- Return one complete Verilog/SystemVerilog module only, inside a ```verilog block.

SPECIFICATION:
{spec}

CURRENT VERILOG:
```verilog
{code}
```

CROSS-VERIFICATION EVIDENCE:
{mismatch_text}

SPEC-ONLY WITNESS EVIDENCE:
{witness_text}

DETERMINISTIC LOCALIZATION ({localization.get('method')}):
{localization.get('snippet','')}

Think privately, then output only the repaired complete module.
"""


def build_python_repair_prompt(
    spec: str, code: str, mismatch_text: str, witness_text: str, runtime_error: Optional[str]
) -> str:
    if runtime_error:
        task = f"Fix the Python execution/modeling problem first: {runtime_error}"
    else:
        task = "Repair the Python hardware-reference behavior indicated by the evidence below."
    return f"""You are the Python hardware-reference repair agent in a code-blind verification workflow.
{task}

NON-NEGOTIABLE RULES:
- The natural-language specification is authoritative.
- You cannot see the Verilog peer implementation; do not blindly copy peer behavior.
- Preserve exact port names, bit widths, cycle semantics, reset semantics, and two's-complement behavior.
- Make the smallest justified change; preserve already-correct behavior.
- Return one complete executable Python reference model only, inside a ```python block.

SPECIFICATION:
{spec}

CURRENT PYTHON REFERENCE:
```python
{code}
```

CROSS-VERIFICATION EVIDENCE:
{mismatch_text}

SPEC-ONLY WITNESS EVIDENCE:
{witness_text}

Think privately, then output only the repaired complete Python model.
"""


# ------------------------------ Active checkpoint ------------------------------

def active_state_path(cfg: Config) -> Path:
    return Path(cfg.out_dir) / "active_state.json"


def results_path(cfg: Config) -> Path:
    return Path(cfg.out_dir) / "results.jsonl"


def checkpoint_zip_path(cfg: Config) -> Path:
    out = Path(cfg.out_dir)
    return out.parent / f"{out.name}_checkpoint.zip"


def refresh_checkpoint_zip(cfg: Config) -> None:
    """Create a portable checkpoint archive for the next Kaggle session."""
    out = Path(cfg.out_dir)
    if not out.exists():
        return
    target = checkpoint_zip_path(cfg)
    tmp_base = target.with_suffix("")
    # make_archive writes <base>.zip. Remove the previous archive only after
    # a new temporary archive has been created successfully.
    temp_base = str(tmp_base) + ".tmp"
    temp_zip = Path(temp_base + ".zip")
    if temp_zip.exists():
        temp_zip.unlink()
    shutil.make_archive(temp_base, "zip", root_dir=str(out.parent), base_dir=out.name)
    os.replace(temp_zip, target)


def save_active(cfg: Config, state: Dict[str, Any]) -> None:
    atomic_json(active_state_path(cfg), state)
    # The JSON file is the authoritative live checkpoint. Keep a portable zip
    # synchronized too, so a Kaggle output can be attached to the next session.
    refresh_checkpoint_zip(cfg)


def clear_active(cfg: Config) -> None:
    p = active_state_path(cfg)
    if p.exists():
        p.unlink()


def load_completed(cfg: Config) -> Dict[str, Dict[str, Any]]:
    out = {}
    p = results_path(cfg)
    if p.exists():
        for line in p.read_text(encoding="utf-8", errors="replace").splitlines():
            try:
                obj = json.loads(line)
                out[obj["task_id"]] = obj
            except Exception:
                pass
    return out


def prepare_resume(cfg: Config) -> None:
    out = Path(cfg.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    if not cfg.resume_from:
        return
    src = Path(cfg.resume_from)
    if not src.exists():
        print(f"[WARN] resume_from does not exist: {src}")
        return
    for name in ("results.jsonl", "active_state.json"):
        s = src / name
        d = out / name
        if s.exists() and not d.exists():
            shutil.copy2(s, d)
    src_candidates = src / "candidates"
    dst_candidates = out / "candidates"
    if src_candidates.exists() and not dst_candidates.exists():
        shutil.copytree(src_candidates, dst_candidates)


# ------------------------------ Main robust problem loop ------------------------------

def run_triage_problem(
    task: Task, cm: dict, v_backend, p_backend, cfg: Config, guard: SessionGuard
) -> Dict[str, Any]:
    guard.check("starting problem")
    t0 = time.monotonic()
    state_file = active_state_path(cfg)
    restored = None
    if state_file.exists():
        try:
            cand = json.loads(state_file.read_text(encoding="utf-8"))
            if cand.get("task_id") == task.task_id:
                restored = cand
        except Exception:
            restored = None

    if restored:
        print(f"  [resume] restoring {task.task_id} from repair round {restored.get('round_completed', 0)}")
        v_pool = restored["v_pool"]
        p_pool = restored["p_pool"]
        witnesses = [Witness(**w) for w in restored.get("witnesses", [])]
        witness_meta = restored.get("witness_meta", {})
        completions = int(restored.get("llm_completions", 0))
        round_completed = int(restored.get("round_completed", 0))
        route_history = restored.get("route_history", [])
        best_snapshot = restored.get("best_snapshot")
    else:
        guard.check("initial generation")
        v_prompt = cm["build_v_prompt"](task.question, 0, v_history=[], v_error="")
        p_prompt = cm["build_p_prompt"](task.question, 0, p_history=[], p_error="")
        with ThreadPoolExecutor(max_workers=2) as ex:
            fv = ex.submit(v_backend.generate, v_prompt, cfg.k_initial, cfg.code_temperature, cfg.code_max_tokens)
            fp = ex.submit(p_backend.generate, p_prompt, cfg.k_initial, cfg.code_temperature, cfg.code_max_tokens)
            v_raw = fv.result()
            p_raw = fp.result()
        v_pool = dedup_codes([cm["extract_verilog"](x or "") or "" for x in v_raw])
        p_pool = dedup_codes([cm["extract_python"](x or "") or "" for x in p_raw])
        completions = cfg.k_initial * 2
        if not v_pool or not p_pool:
            raise RuntimeError("Initial ChipMATE extraction produced an empty candidate pool")

        # Spec-only witnesses use fixed canonical sequences and two independent agents.
        guard.check("spec witness generation")
        witnesses, used, witness_meta = generate_consensus_witnesses(
            cm, task.question, task.ref_stub, v_backend, p_backend, cfg
        )
        completions += used
        round_completed = 0
        route_history = []
        best_snapshot = None
        save_active(cfg, {
            "task_id": task.task_id,
            "v_pool": v_pool,
            "p_pool": p_pool,
            "witnesses": [asdict(w) for w in witnesses],
            "witness_meta": witness_meta,
            "llm_completions": completions,
            "round_completed": round_completed,
            "route_history": route_history,
            "best_snapshot": best_snapshot,
        })

    stagnation = 0
    # round_completed means repair rounds already completed. We still evaluate the
    # current pool before deciding whether another repair is needed.
    while True:
        guard.check(f"evidence evaluation for {task.task_id}")
        stimuli = build_regression_bank(cm, task.ref_stub, cfg, round_completed)
        ev = evaluate_matrix(cm, task.ref_stub, v_pool, p_pool, stimuli, cfg.workers)
        vi, pj = ev.best_i, ev.best_j
        best_v, best_p = v_pool[vi], p_pool[pj]

        v_w, v_w_details, v_w_err = evaluate_witnesses_on_code(cm, task.ref_stub, best_v, witnesses, "v")
        p_w, p_w_details, p_w_err = evaluate_witnesses_on_code(cm, task.ref_stub, best_p, witnesses, "p")
        witnesses_exist = bool(witnesses)
        score = combined_score(ev, v_w, p_w, witnesses_exist, cfg)

        current = {
            "v": best_v,
            "p": best_p,
            "score": score,
            "pair_match": ev.best_pair_match,
            "v_support": ev.v_support[vi],
            "p_support": ev.p_support[pj],
            "v_witness": v_w,
            "p_witness": p_w,
            "v_error": ev.v_sims[vi].error,
            "p_error": ev.p_sims[pj].error,
            "round": round_completed,
        }

        # Best-so-far snapshot is immutable unless evidence improves on the SAME
        # current regression bank. If bank grew, re-evaluate old best for fair comparison.
        if best_snapshot is None:
            best_snapshot = current
        else:
            old_v = best_snapshot["v"]
            old_p = best_snapshot["p"]
            old_ev = evaluate_matrix(cm, task.ref_stub, [old_v], [old_p], stimuli, min(2, cfg.workers))
            old_vw, _, _ = evaluate_witnesses_on_code(cm, task.ref_stub, old_v, witnesses, "v")
            old_pw, _, _ = evaluate_witnesses_on_code(cm, task.ref_stub, old_p, witnesses, "p")
            old_score = combined_score(old_ev, old_vw, old_pw, witnesses_exist, cfg)
            old_witness_sum = old_vw + old_pw
            new_witness_sum = v_w + p_w

            # Evidence gate: strictly better, and never regress spec-only consensus
            # witnesses by more than floating noise.
            if score > old_score + cfg.min_improvement and new_witness_sum + 1e-9 >= old_witness_sum:
                best_snapshot = current
                stagnation = 0
            else:
                # Backtrack by restoring the previous best artifacts to the pool.
                if old_v not in v_pool:
                    v_pool.append(old_v)
                if old_p not in p_pool:
                    p_pool.append(old_p)
                stagnation += 1
                current = dict(best_snapshot)
                best_v, best_p = old_v, old_p
                # Recompute a minimal 1x1 evidence view for stopping/routing if needed.
                ev = old_ev
                vi = pj = 0
                v_w, p_w = old_vw, old_pw
                v_w_details, p_w_details = [], []
                score = old_score

        # Strong conservative stop. If no witness exists, require one extra repair
        # opportunity unless the pool consensus is exceptionally high.
        support_ok = current["v_support"] >= cfg.strong_support_threshold and current["p_support"] >= cfg.strong_support_threshold
        witness_ok = (not witnesses_exist) or (current["v_witness"] >= 1.0 and current["p_witness"] >= 1.0)
        pair_ok = current["pair_match"] >= cfg.strong_pair_threshold
        if pair_ok and support_ok and witness_ok:
            status = "strong_evidence"
            break

        if round_completed >= cfg.max_repair_rounds:
            status = "max_rounds"
            break
        if completions >= cfg.fair_completion_budget:
            status = "completion_budget"
            break

        route = route_responsibility(ev, v_w, p_w, witnesses_exist, cfg)
        mm = ev.first_mismatch
        minimized = minimize_counterexample(
            cm, task.ref_stub, best_v, best_p, stimuli, mm, cfg.ddmin_calls
        ) if mm else {"steps": [], "mismatch": None, "calls": 0}
        mismatch_text = format_mismatch(mm, minimized)
        failing_signal = (mm or {}).get("signal") or next(iter(dict(cm["parse_ports"](task.ref_stub)[1])), "")
        localization = yosys_localize(best_v, str(failing_signal), cfg.yosys_depth, cfg.max_localized_lines)

        # Route-specific witness evidence.
        v_w_text = witness_feedback(v_w_details)
        p_w_text = witness_feedback(p_w_details)

        remaining = cfg.fair_completion_budget - completions
        if remaining <= 0:
            status = "completion_budget"
            break

        # Dynamic allocation: spend completions only on the suspected weak side;
        # ambiguous cases receive two candidates from each side. Never exceed budget.
        new_v: List[str] = []
        new_p: List[str] = []
        if route == "verilog":
            nv = min(3, remaining)
            if nv > 0:
                guard.check("Verilog repair generation")
                prompt = build_verilog_repair_prompt(
                    task.question, best_v, mismatch_text, localization,
                    v_w_text, current.get("v_error")
                )
                outs = v_backend.generate(prompt, nv, cfg.repair_temperature, cfg.code_max_tokens)
                new_v = [cm["extract_verilog"](x or "") or "" for x in outs]
                completions += nv
        elif route == "python":
            np_ = min(3, remaining)
            if np_ > 0:
                guard.check("Python repair generation")
                prompt = build_python_repair_prompt(
                    task.question, best_p, mismatch_text,
                    p_w_text, current.get("p_error")
                )
                outs = p_backend.generate(prompt, np_, cfg.repair_temperature, cfg.code_max_tokens)
                new_p = [cm["extract_python"](x or "") or "" for x in outs]
                completions += np_
        else:
            nv = min(2, remaining)
            np_ = min(2, max(0, remaining - nv))
            guard.check("dual repair generation")
            if nv:
                pv = build_verilog_repair_prompt(
                    task.question, best_v, mismatch_text, localization,
                    v_w_text, current.get("v_error")
                )
                new_v = [cm["extract_verilog"](x or "") or "" for x in v_backend.generate(
                    pv, nv, cfg.repair_temperature, cfg.code_max_tokens
                )]
                completions += nv
            if np_:
                pp = build_python_repair_prompt(
                    task.question, best_p, mismatch_text,
                    p_w_text, current.get("p_error")
                )
                new_p = [cm["extract_python"](x or "") or "" for x in p_backend.generate(
                    pp, np_, cfg.repair_temperature, cfg.code_max_tokens
                )]
                completions += np_

        old_v_count, old_p_count = len(v_pool), len(p_pool)
        v_pool = dedup_codes(v_pool + new_v)
        p_pool = dedup_codes(p_pool + new_p)
        route_history.append({
            "round": round_completed + 1,
            "route": route,
            "score_before": score,
            "pair_match": current["pair_match"],
            "v_support": current["v_support"],
            "p_support": current["p_support"],
            "v_witness": current["v_witness"],
            "p_witness": current["p_witness"],
            "mismatch": mm,
            "minimized_steps": minimized.get("steps", []),
            "localization_method": localization.get("method"),
            "new_v_candidates": len(v_pool) - old_v_count,
            "new_p_candidates": len(p_pool) - old_p_count,
            "llm_completions": completions,
        })
        round_completed += 1

        # If two consecutive rounds add nothing useful, stop rather than spin.
        if len(v_pool) == old_v_count and len(p_pool) == old_p_count:
            stagnation += 1
        if stagnation >= 2:
            status = "stagnation"
            break

        save_active(cfg, {
            "task_id": task.task_id,
            "v_pool": v_pool,
            "p_pool": p_pool,
            "witnesses": [asdict(w) for w in witnesses],
            "witness_meta": witness_meta,
            "llm_completions": completions,
            "round_completed": round_completed,
            "route_history": route_history,
            "best_snapshot": best_snapshot,
        })

    final_v = best_snapshot["v"] if best_snapshot else current["v"]
    final_p = best_snapshot["p"] if best_snapshot else current["p"]
    final_v = rename_first_module(final_v, task.target_module)

    # Blind official evaluation; this result is NEVER fed back into the loop.
    official = evaluate_official_rtllm(task, final_v, cfg)

    cand_dir = Path(cfg.out_dir) / "candidates"
    cand_dir.mkdir(parents=True, exist_ok=True)
    (cand_dir / f"{task.task_id}.v").write_text(final_v, encoding="utf-8")
    (cand_dir / f"{task.task_id}.py").write_text(final_p, encoding="utf-8")

    result = {
        "task_id": task.task_id,
        "folder": task.folder,
        "status": status,
        "llm_completions": completions,
        "repair_rounds": round_completed,
        "witness_meta": witness_meta,
        "witnesses": [asdict(w) for w in witnesses],
        "route_history": route_history,
        "final_evidence": best_snapshot,
        "final_verilog_sha": sha(final_v),
        "final_python_sha": sha(final_p),
        "official_eval": official,
        "elapsed_sec": time.monotonic() - t0,
    }
    return result


# ------------------------------ Blind official RTLLM evaluation ------------------------------

def _tb_top_module(tb_text: str) -> Optional[str]:
    # Prefer the conventional RTLLM name, otherwise take the first module declared
    # in the testbench file. This is evaluation-only and is never shown to agents.
    mods = re.findall(r"\bmodule\s+([A-Za-z_]\w*)", tb_text or "")
    if "testbench" in mods:
        return "testbench"
    return mods[0] if mods else None


def _run_official_pair(tb_path: Path, rtl_path: Path, timeout: int) -> Dict[str, Any]:
    """Compile/run an RTLLM testbench in its own folder so relative data files work."""
    tb_text = tb_path.read_text(encoding="utf-8", errors="replace")
    top = _tb_top_module(tb_text)
    with tempfile.TemporaryDirectory() as td:
        simv = Path(td) / "simv"
        cmd = ["iverilog", "-g2012"]
        if top:
            cmd += ["-s", top]
        cmd += ["-I", str(tb_path.parent), "-o", str(simv), str(tb_path), str(rtl_path)]
        try:
            cp = subprocess.run(
                cmd, cwd=str(tb_path.parent), capture_output=True, text=True, timeout=timeout
            )
        except subprocess.TimeoutExpired:
            return {"compile": False, "functional_pass": False, "error": "compile_timeout"}
        if cp.returncode != 0:
            return {
                "compile": False,
                "functional_pass": False,
                "error": "compile_error",
                "stderr": (cp.stderr or "")[-3000:],
            }
        try:
            rp = subprocess.run(
                ["vvp", str(simv)], cwd=str(tb_path.parent), capture_output=True, text=True, timeout=timeout
            )
        except subprocess.TimeoutExpired:
            return {"compile": True, "functional_pass": False, "error": "sim_timeout"}
        text = (rp.stdout or "") + "\n" + (rp.stderr or "")
        passed = "Your Design Passed" in text
        if not passed:
            m = re.search(r"Test completed with\s+(\d+)\s*/\s*(\d+)\s+failures", text, re.I)
            if m and int(m.group(1)) == 0:
                passed = True
        return {
            "compile": True,
            "functional_pass": bool(passed and rp.returncode == 0),
            "returncode": rp.returncode,
            "output_tail": text[-3000:],
        }


def evaluate_official_rtllm(task: Task, candidate_sv: str, cfg: Config) -> Dict[str, Any]:
    """Blind final evaluation. Golden RTL is used only to sanity-check the harness.

    The reference implementation is never read by the inference/repair loop. The
    preflight happens only after the candidate is frozen; if the benchmark's own
    reference cannot pass under the local Icarus setup, the case is marked
    evaluator_unsupported instead of unfairly counting it as a model failure.
    """
    tb_path = Path(task.testbench_path)
    ref_path = Path(task.reference_path)
    if not tb_path.exists():
        return {"evaluable": False, "compile": None, "functional_pass": None, "error": "missing_testbench"}
    if not ref_path.exists():
        return {"evaluable": False, "compile": None, "functional_pass": None, "error": "missing_reference_for_preflight"}

    preflight = _run_official_pair(tb_path, ref_path, cfg.official_eval_timeout_sec)
    if not preflight.get("functional_pass"):
        return {
            "evaluable": False,
            "compile": None,
            "functional_pass": None,
            "error": "evaluator_unsupported",
            "reference_preflight": preflight,
        }

    with tempfile.TemporaryDirectory() as td:
        dut = Path(td) / "candidate.v"
        dut.write_text(candidate_sv, encoding="utf-8")
        got = _run_official_pair(tb_path, dut, cfg.official_eval_timeout_sec)
    got["evaluable"] = True
    got["reference_preflight_passed"] = True
    return got


# ------------------------------ Experiment driver ------------------------------

def summarize_results(cfg: Config) -> Dict[str, Any]:
    rows = list(load_completed(cfg).values())
    if not rows:
        return {"n": 0}
    eval_rows = [r for r in rows if r.get("official_eval", {}).get("evaluable") is True]
    comp = [r.get("official_eval", {}).get("compile") is True for r in eval_rows]
    func = [r.get("official_eval", {}).get("functional_pass") is True for r in eval_rows]
    errors = [r.get("official_eval", {}).get("error") for r in rows]
    summary = {
        "n_completed": len(rows),
        "n_evaluable": len(eval_rows),
        "n_evaluator_unsupported": len(rows) - len(eval_rows),
        "compile_rate_evaluable": (sum(comp) / len(eval_rows)) if eval_rows else None,
        "functional_pass_rate_evaluable": (sum(func) / len(eval_rows)) if eval_rows else None,
        # Conservative all-completed denominator is also recorded; unsupported
        # harness cases count as non-passes here, so nothing is hidden.
        "functional_pass_rate_all_completed": sum(
            r.get("official_eval", {}).get("functional_pass") is True for r in rows
        ) / len(rows),
        "compiled_evaluable": sum(comp),
        "functional_passed_evaluable": sum(func),
        "avg_llm_completions": sum(r.get("llm_completions", 0) for r in rows) / len(rows),
        "avg_repair_rounds": sum(r.get("repair_rounds", 0) for r in rows) / len(rows),
        "official_eval_errors": {e: errors.count(e) for e in set(errors) if e},
    }
    atomic_json(Path(cfg.out_dir) / "summary.json", summary)
    return summary


def run_experiment(cfg: Config = CFG) -> None:
    prepare_resume(cfg)
    cm = import_chipmate()
    v_backend = cm["make_backend"](
        provider="openai-compat", model=cfg.v_model,
        base_url=cfg.v_base_url, api_key="dummy", timeout=900, max_retries=4
    )
    p_backend = cm["make_backend"](
        provider="openai-compat", model=cfg.p_model,
        base_url=cfg.p_base_url, api_key="dummy", timeout=900, max_retries=4
    )
    tasks = discover_rtllm(cfg.rtllm_root)
    completed = load_completed(cfg)
    guard = SessionGuard(cfg)
    consecutive_errors = 0
    print(
        f"Discovered {len(tasks)} RTLLM tasks; already completed {len(completed)}; "
        f"total session elapsed={guard.elapsed()/60:.1f} min"
    )

    for idx, task in enumerate(tasks):
        if task.task_id in completed:
            continue
        try:
            guard.check(f"problem {task.task_id}", include_next_problem=True)
        except TimeBudgetExceeded as e:
            print(f"\n[TIME GUARD] {e}")
            refresh_checkpoint_zip(cfg)
            break
        print(f"\n[{idx+1}/{len(tasks)}] {task.folder}")
        t0 = time.monotonic()
        try:
            result = run_triage_problem(task, cm, v_backend, p_backend, cfg, guard)
            atomic_jsonl_append(results_path(cfg), result)
            clear_active(cfg)
            refresh_checkpoint_zip(cfg)
            elapsed = time.monotonic() - t0
            guard.record_problem(elapsed)
            consecutive_errors = 0
            print(
                f"  official: compile={result['official_eval'].get('compile')} "
                f"pass={result['official_eval'].get('functional_pass')} "
                f"rounds={result['repair_rounds']} completions={result['llm_completions']} "
                f"time={elapsed/60:.1f}m"
            )
            print("  running summary:", summarize_results(cfg))
        except TimeBudgetExceeded as e:
            # Active state was saved after the last completed repair round.
            print(f"\n[TIME GUARD] {e}")
            print("Active task state is preserved in active_state.json. Stop and commit notebook outputs.")
            refresh_checkpoint_zip(cfg)
            break
        except Exception as e:
            # Preserve an error record so one pathological task cannot kill the run.
            err = {
                "task_id": task.task_id,
                "folder": task.folder,
                "status": "pipeline_exception",
                "error": repr(e),
                "elapsed_sec": time.monotonic() - t0,
                "official_eval": {"compile": False, "functional_pass": False, "error": "pipeline_exception"},
            }
            atomic_jsonl_append(results_path(cfg), err)
            clear_active(cfg)
            guard.record_problem(time.monotonic() - t0)
            consecutive_errors += 1
            print(
                f"  [ERROR] {e!r}; recorded "
                f"({consecutive_errors}/{cfg.max_consecutive_pipeline_errors} consecutive)"
            )
            refresh_checkpoint_zip(cfg)
            if consecutive_errors >= cfg.max_consecutive_pipeline_errors:
                print(
                    "  [CIRCUIT BREAKER] Repeated pipeline errors suggest a systemic "
                    "setup/API problem. Stopping with a resumable checkpoint."
                )
                break

    print("\nFINAL / CURRENT SUMMARY")
    print(json.dumps(summarize_results(cfg), indent=2))
    refresh_checkpoint_zip(cfg)
    print(f"Checkpoint directory: {cfg.out_dir}")
    print(f"Portable checkpoint: {checkpoint_zip_path(cfg)}")


if __name__ == "__main__":
    run_experiment(CFG)


In [ ]:
# 5) Configure and run TRIAGE-RTL.
# RESUME_FROM:
#   - None for a fresh run.
#   - A directory from a previous Kaggle output, e.g.
#       '/kaggle/input/triage-rtl-run1/triage_rtllm'
#   - Or the extracted directory from the portable checkpoint ZIP.

import sys
sys.path.insert(0, '/kaggle/working')
from triage_rtl_kaggle import CFG, discover_rtllm, run_experiment

RESUME_FROM = None

CFG.rtllm_root = '/kaggle/working/RTLLM'
CFG.out_dir = '/kaggle/working/triage_rtllm'
CFG.resume_from = RESUME_FROM
CFG.v_model = V_MODEL
CFG.p_model = P_MODEL
CFG.v_base_url = 'http://127.0.0.1:8001/v1'
CFG.p_base_url = 'http://127.0.0.1:8002/v1'

# ChipMATE n=3,t=3 spends 18 model completions/problem (9 Verilog + 9 Python).
# Keep the same hard cap and allocate it adaptively.
CFG.fair_completion_budget = 18
CFG.k_initial = 3
CFG.max_repair_rounds = 3
CFG.session_budget_hours = 11.25
CFG.stop_margin_minutes = 25
CFG.session_start_epoch = TRIAGE_NOTEBOOK_START_EPOCH
CFG.max_consecutive_pipeline_errors = 3

# Robust verification defaults. Increase only after the first end-to-end run.
CFG.random_seeds = (41, 137, 271)
CFG.random_per_seed = 96
CFG.directed_limit = 48
CFG.fresh_random_per_round = 64
CFG.max_regression_vectors = 512

# Sanity-check dataset discovery before consuming GPU inference time.
tasks = discover_rtllm(CFG.rtllm_root)
print('RTLLM tasks discovered:', len(tasks))
print('First 5:', [t.folder for t in tasks[:5]])
if len(tasks) < 45:
    raise RuntimeError('Too few RTLLM tasks were discovered; inspect the pinned repository layout before running.')

run_experiment(CFG)


In [ ]:
# 6) Inspect current results and checkpoint artifacts.
import json
from pathlib import Path

out = Path('/kaggle/working/triage_rtllm')
summary = out / 'summary.json'
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), indent=2))

print('\nGenerated candidates:', len(list((out/'candidates').glob('*.v'))) if (out/'candidates').exists() else 0)
print('Live state:', out/'active_state.json')
print('Completed results:', out/'results.jsonl')
print('Portable checkpoint:', Path('/kaggle/working/triage_rtllm_checkpoint.zip'))


## Resume in the next Kaggle session

The code **cannot automatically create a new Kaggle session** after the platform stops the current one. It does everything that can be done inside the notebook:

1. after every repair round it writes `active_state.json`;
2. after every completed RTLLM problem it appends `results.jsonl`;
3. it refreshes `/kaggle/working/triage_rtllm_checkpoint.zip`;
4. before the configured wall-clock budget expires it stops launching new work.

Then use Kaggle **Save Version** so `/kaggle/working` becomes notebook output. In the next session attach that output as an input and set `RESUME_FROM` in Cell 5 to the previous `triage_rtllm` directory. Completed problems are skipped, and an interrupted problem resumes from its last saved repair round.

### Fairness rule
The official RTLLM testbench and golden implementation are **not available to generation, routing, localization, repair, or acceptance**. The golden RTL body is never read by inference; only its module declaration is extracted to obtain the benchmark interface. After the final Verilog is frozen, the official testbench is run once for scoring. The golden design is used at that point only to preflight that the local Icarus evaluator itself can execute the benchmark case.
